# blur-1 — inference

Generate new paintings from the trained model. Nothing trains here: this notebook
rebuilds the network, loads the EMA weights from `checkpoints/latest.pt`, and samples.

**Run all cells once**, then edit the knobs in **section 4** and re-run just that cell.
Every result is also saved to `samples/inference/`.

| knob | values |
|---|---|
| artist | `van_gogh`, `renoir`, `monet`, `cezanne`, or `None` (any) |
| genre | `landscape`, `portrait`, `still_life`, `cityscape`, `genre_painting`, `other`, or `None` |
| seed | same seed → same paintings; new seed → new paintings |
| guidance | 1–2 varied but muddy · 3 balanced · 5+ vivid, over-saturated |

Speed: about 1 second per painting at 50 steps on an M2 Pro.

In [ ]:
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from torchvision.utils import make_grid

DEVICE  = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
ROOT    = Path(".").resolve()
CKPT    = ROOT / "checkpoints" / "latest.pt"
OUT_DIR = ROOT / "samples" / "inference"; OUT_DIR.mkdir(parents=True, exist_ok=True)

assert CKPT.exists(), f"no checkpoint at {CKPT} - train with painting_diffusion.ipynb first"
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Config

These values define the network's shape and label ids. They **must match**
`painting_diffusion.ipynb`, or the checkpoint will not load.

In [ ]:
ARTISTS = {22: "van_gogh", 17: "renoir", 4: "monet", 16: "cezanne"}   # WikiArt label id -> name
ARTIST_NAMES = [ARTISTS[k] for k in sorted(ARTISTS)]                   # order = embedding index
ARTIST_ID = {n: i for i, n in enumerate(ARTIST_NAMES)}

GENRES = ["landscape", "portrait", "still_life", "cityscape", "genre_painting", "other"]
GENRE_ID = {n: i for i, n in enumerate(GENRES)}

N_ARTISTS, N_GENRES = len(ARTIST_NAMES), len(GENRES)
NULL_ARTIST, NULL_GENRE = N_ARTISTS, N_GENRES    # extra embedding slots = "no condition"

IMAGE_SIZE   = 64
BASE_CH      = 64
CHANNEL_MULT = (1, 2, 3, 4)
NUM_RES      = 2
ATTN_RES     = (16, 8)
NUM_HEADS    = 4
DROPOUT      = 0.1      # inactive at inference (model.eval())

TIMESTEPS = 1000
PRED_TYPE = "v"

## 2. Network

Copied unchanged from section 6 of the training notebook.

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float()[:, None] * freqs[None]
    return torch.cat([torch.cos(a), torch.sin(a)], dim=-1)

def gn(ch):
    g = 32                      # largest group count <= 32 that divides ch
    while ch % g:
        g //= 2
    return nn.GroupNorm(g, ch)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb, dropout):
        super().__init__()
        self.norm1 = gn(cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.emb   = nn.Linear(emb, cout * 2)          # FiLM scale + shift
        self.norm2 = gn(cout)
        self.drop  = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip  = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.emb(F.silu(emb))[:, :, None, None].chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.drop(F.silu(h)))
        return h + self.skip(x)

class AttnBlock(nn.Module):
    def __init__(self, ch, heads):
        super().__init__()
        self.norm, self.heads = gn(ch), heads
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(
            B, 3, self.heads, C // self.heads, H * W).unbind(1)
        q, k, v = (t.transpose(-1, -2) for t in (q, k, v))     # B,heads,HW,dim
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(-1, -2).reshape(B, C, H, W)
        return x + self.proj(h)

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.op = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.op(F.interpolate(x, scale_factor=2, mode="nearest"))

class Stage(nn.Module):
    def __init__(self, res, attn=None):
        super().__init__(); self.res, self.attn = res, attn
    def forward(self, x, emb):
        x = self.res(x, emb)
        return self.attn(x) if self.attn is not None else x

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        emb = BASE_CH * 4
        self.emb_dim = emb
        self.time_mlp = nn.Sequential(nn.Linear(BASE_CH, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.artist_emb = nn.Embedding(N_ARTISTS + 1, emb)
        self.genre_emb  = nn.Embedding(N_GENRES + 1, emb)

        self.stem = nn.Conv2d(3, BASE_CH, 3, padding=1)
        self.down, chans, ch, res = nn.ModuleList(), [BASE_CH], BASE_CH, IMAGE_SIZE
        for i, m in enumerate(CHANNEL_MULT):
            out = BASE_CH * m
            for _ in range(NUM_RES):
                self.down.append(Stage(ResBlock(ch, out, emb, DROPOUT),
                                       AttnBlock(out, NUM_HEADS) if res in ATTN_RES else None))
                ch = out; chans.append(ch)
            if i != len(CHANNEL_MULT) - 1:
                self.down.append(Down(ch)); chans.append(ch); res //= 2

        self.mid1 = ResBlock(ch, ch, emb, DROPOUT)
        self.mid_attn = AttnBlock(ch, NUM_HEADS)
        self.mid2 = ResBlock(ch, ch, emb, DROPOUT)

        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(CHANNEL_MULT))):
            out = BASE_CH * m
            for j in range(NUM_RES + 1):
                self.up.append(Stage(ResBlock(ch + chans.pop(), out, emb, DROPOUT),
                                     AttnBlock(out, NUM_HEADS) if res in ATTN_RES else None))
                ch = out
                if i and j == NUM_RES:
                    self.up.append(Up(ch)); res *= 2

        self.out = nn.Sequential(gn(ch), nn.SiLU(), nn.Conv2d(ch, 3, 3, padding=1))

    def forward(self, x, t, artist, genre):
        emb = (self.time_mlp(timestep_embedding(t, BASE_CH))
               + self.artist_emb(artist) + self.genre_emb(genre))
        h = self.stem(x); hs = [h]
        for mod in self.down:
            h = mod(h, emb) if isinstance(mod, Stage) else mod(h)
            hs.append(h)
        h = self.mid2(self.mid_attn(self.mid1(h, emb)), emb)
        for mod in self.up:
            if isinstance(mod, Stage):
                h = mod(torch.cat([h, hs.pop()], dim=1), emb)
            else:
                h = mod(h)
        return self.out(h)

## 3. Load the trained weights

Loads the **EMA** weights, the averaged copy that training sampled from. The raw
weights in the same file make visibly worse paintings.

In [ ]:
ck = torch.load(CKPT, map_location="cpu")
model = UNet().to(DEVICE)
model.load_state_dict({k: v.to(DEVICE) for k, v in ck["ema"].items()})   # strict: shapes must match
model.eval()
TRAINED_STEPS = ck["step"]
del ck                                   # drop optimizer state, ~2/3 of the file

nparam = sum(p.numel() for p in model.parameters())
print(f"loaded EMA weights from step {TRAINED_STEPS:,} | {nparam/1e6:.1f}M parameters")

In [ ]:
# ---- noise schedule (cosine, same as training) ---------------------------
def cosine_alpha_bar(T, s=0.008):
    t = torch.linspace(0, T, T + 1, dtype=torch.float64) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0]).clamp(1e-8, 1.0)

ab_full = cosine_alpha_bar(TIMESTEPS)
betas = (1 - ab_full[1:] / ab_full[:-1]).clamp(0, 0.999)
alphas_bar = torch.cumprod(1 - betas, 0).float().to(DEVICE)

# ---- DDIM sampler with two-axis classifier-free guidance -------------------
@torch.no_grad()
def sample(n=8, artist=None, genre=None, w_artist=3.0, w_genre=2.0,
           steps=50, eta=0.0, seed=None, artist_mix=None, net=None):
    # artist / genre: a name, or None for "any". artist_mix: e.g. {"monet": 0.5, "van_gogh": 0.5}
    if artist is not None and artist not in ARTIST_ID:
        raise ValueError(f"artist must be one of {ARTIST_NAMES} or None, got {artist!r}")
    if genre is not None and genre not in GENRE_ID:
        raise ValueError(f"genre must be one of {GENRES} or None, got {genre!r}")
    for name in (artist_mix or {}):
        if name not in ARTIST_ID:
            raise ValueError(f"artist_mix names must be in {ARTIST_NAMES}, got {name!r}")

    net = net or model
    if seed is not None:
        torch.manual_seed(seed)

    a_id = NULL_ARTIST if artist is None else ARTIST_ID[artist]
    g_id = NULL_GENRE  if genre  is None else GENRE_ID[genre]
    A_null = torch.full((n,), NULL_ARTIST, device=DEVICE, dtype=torch.long)
    G_null = torch.full((n,), NULL_GENRE,  device=DEVICE, dtype=torch.long)
    A_cond = torch.full((n,), a_id, device=DEVICE, dtype=torch.long)
    G_cond = torch.full((n,), g_id, device=DEVICE, dtype=torch.long)

    ts = torch.linspace(TIMESTEPS - 1, 0, steps).long().to(DEVICE)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)

    for i, t in enumerate(tqdm(ts, desc="painting", leave=False)):
        tb = t.repeat(n)
        x3 = torch.cat([x, x, x], 0)
        t3 = torch.cat([tb, tb, tb], 0)
        a3 = torch.cat([A_null, A_cond, A_cond], 0)
        g3 = torch.cat([G_null, G_null, G_cond], 0)

        if artist_mix:
            out = 0
            for name, wt in artist_mix.items():
                a3m = a3.clone()
                a3m[n:] = ARTIST_ID[name]
                out = out + wt * net(x3, t3, a3m, g3)
            pred = out / sum(artist_mix.values())
        else:
            pred = net(x3, t3, a3, g3)

        p_none, p_art, p_both = pred.chunk(3, 0)
        p = p_none + w_artist * (p_art - p_none) + w_genre * (p_both - p_art)

        ab_t = alphas_bar[t]
        ab_prev = alphas_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=DEVICE)
        a_t, b_t = ab_t.sqrt(), (1 - ab_t).sqrt()

        if PRED_TYPE == "v":
            x0 = a_t * x - b_t * p
            eps = b_t * x + a_t * p
        else:
            eps = p
            x0 = (x - b_t * eps) / a_t
        x0 = x0.clamp(-1, 1)

        sigma = eta * ((1 - ab_prev) / (1 - ab_t)).sqrt() * (1 - ab_t / ab_prev).sqrt()
        dir_xt = (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        x = ab_prev.sqrt() * x0 + dir_xt
        if eta > 0 and i + 1 < len(ts):
            x = x + sigma * torch.randn_like(x)
    return x

# ---- display / save --------------------------------------------------------
def to_grid(x, nrow):
    g = make_grid(x.detach().float().cpu().clamp(-1, 1).add(1).div(2), nrow=nrow, padding=2)
    return (g.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)

def show(x, nrow=8, title=None):
    g = to_grid(x, nrow)
    rows = math.ceil(len(x) / nrow)
    plt.figure(figsize=(min(16, 1.8 * nrow), 1.8 * rows + 0.4))
    plt.imshow(g); plt.axis("off")
    if title: plt.title(title)
    plt.show()

def save(x, name, nrow=8, upscale=4):
    # writes the native 64px grid plus an upscaled copy for viewing (interpolated, no new detail)
    im = Image.fromarray(to_grid(x, nrow))
    im.save(OUT_DIR / f"{name}.png")
    big = OUT_DIR / f"{name}_x{upscale}.png"
    im.resize((im.width * upscale, im.height * upscale), Image.LANCZOS).save(big)
    return big

print("ready")

## 4. Make a painting

Edit the knobs and re-run this cell. Set `SEED = None` for new paintings on every
run; the seed it used is printed, so you can recreate one you like.

In [ ]:
ARTIST   = "van_gogh"    # van_gogh | renoir | monet | cezanne | None
GENRE    = "landscape"   # landscape | portrait | still_life | cityscape | genre_painting | other | None
N        = 8             # how many paintings
SEED     = 7             # int for repeatable results, None for random
W_ARTIST = 3.0           # how strongly to follow the artist
W_GENRE  = 2.0           # how strongly to follow the genre
STEPS    = 50            # denoising steps; 100-250 is slightly cleaner and slower

seed = SEED if SEED is not None else int(torch.randint(0, 1_000_000, (1,)))
x = sample(N, artist=ARTIST, genre=GENRE, w_artist=W_ARTIST, w_genre=W_GENRE,
           steps=STEPS, seed=seed)
show(x, nrow=8, title=f"{ARTIST} / {GENRE}   seed={seed}")
print("seed:", seed, "| saved:", save(x, f"{ARTIST}__{GENRE}__seed{seed}"))

## 5. Gallery: every painter, same starting noise

Rows are painters, columns are seeds. Each column starts from the **same** random
noise, so the difference down a column comes only from the artist label.

In [ ]:
gallery_genre, cols, gallery_seed = "landscape", 6, 0

rows = [sample(cols, artist=a, genre=gallery_genre, seed=gallery_seed) for a in ARTIST_NAMES]
gallery = torch.cat(rows)
show(gallery, nrow=cols, title=f"{gallery_genre} | rows: {', '.join(ARTIST_NAMES)}")
print("saved:", save(gallery, f"gallery__{gallery_genre}__seed{gallery_seed}", nrow=cols))

## 6. Guidance strength

Same painter, same seed, increasing `w_artist`. Pick the row you like and use that
value in section 4.

In [ ]:
weights = [1.0, 2.0, 3.0, 5.0, 7.0]
sweep = torch.cat([sample(6, artist="monet", genre="landscape", w_artist=w, seed=1) for w in weights])
show(sweep, nrow=6, title="monet / landscape | rows: w_artist = " + ", ".join(map(str, weights)))

## 7. Blend two painters

Each row moves from all Monet to all Van Gogh, with the same seed throughout.

In [ ]:
a, b = "monet", "van_gogh"
mixes = [0.0, 0.25, 0.5, 0.75, 1.0]
blend = torch.cat([sample(6, genre="landscape", seed=3, artist_mix={a: 1 - m, b: m}) for m in mixes])
show(blend, nrow=6, title=f"{a} -> {b} | rows: {b} share = " + ", ".join(map(str, mixes)))
print("saved:", save(blend, f"blend__{a}__{b}", nrow=6))

## 8. Is it new? Closest training painting

Finds the most similar of the 5,202 training paintings for each painting from
section 4 (`x`). It compares at 32×32, across 9 crop offsets and mirrored copies,
because training used random crops and flips.

Judge by eye. A copy shows the **same picture**. A new painting only shares the
palette and style. Matching is by color layout, so the closest painting can be by a
different artist.

In [ ]:
CACHE = ROOT / "data" / "cache_images.npy"
if not CACHE.exists():
    print("skipped: needs data/cache_images.npy from the training notebook")
else:
    imgs = torch.from_numpy(np.load(CACHE)).permute(0, 3, 1, 2)       # uint8 [N,3,80,80]
    span = imgs.shape[-1] - IMAGE_SIZE
    offsets = [0, span // 2, span]

    def small(t):                                                      # [-1,1] -> 32x32 features
        return F.avg_pool2d(t, 2).flatten(1)

    q = small(x.detach().float().cpu())
    best_d = torch.full((len(q),), float("inf"))
    best_img = torch.zeros(len(q), 3, IMAGE_SIZE, IMAGE_SIZE)
    for oy in offsets:
        for ox in offsets:
            crop = imgs[:, :, oy:oy + IMAGE_SIZE, ox:ox + IMAGE_SIZE].float().div(127.5).sub(1)
            for cand in (crop, crop.flip(3)):
                d, i = torch.cdist(q, small(cand)).min(1)
                better = d < best_d
                best_d[better] = d[better]
                best_img[better] = cand[i[better]]

    show(torch.cat([x.detach().float().cpu(), best_img]), nrow=len(x),
         title="top: generated | bottom: closest training painting")
    rms = best_d / math.sqrt(q.shape[1]) / 2                            # RMS gap in [0,1] pixel units
    print("RMS pixel gap to closest training painting:", [f"{v:.3f}" for v in rms.tolist()])